 # **Lab 04 – Spark Streaming**

## **Group: CDHD**

### **Team Members**

| STT | Full Name           | Student ID |
| :-: | ------------------- | :--------: |
|  1  | Phan Thị Phương Chi |  23120025  |
|  2  | Trần Thanh Đạt      |  23120030  |
|  3  | Phạm Ngọc Duy       |  23120035  |
|  4  | Lê Hoàng Mỹ Hạ      |  23120038  |

## **Task 1: Repository Cloning and File Discovery**

### Mục tiêu
- Khám phá toàn bộ kho mã nguồn Python được chọn làm **corpus** cho CPG Parser Service (Task 2).
- Kết quả của task này là danh sách file `.py` đã phân loại, lưu vào `output/discovered_files.csv`. 
  
### Thông tin repo

| Trường        | Giá trị |
|:--------------|:--------|
| **Tên repo**  | `peft` |
| **Tổ chức**   | `huggingface` |
| **URL clone** | `https://github.com/huggingface/peft.git` |

### 0. Imports & cấu hình đường dẫn

In [9]:
import os
import sys
import subprocess
import platform
from pathlib import Path

import pandas as pd

# resolve đường dẫn repo: hỗ trợ cả wsl và windows native
WSL_REPO_PATH   = "/mnt/e/BD/peft"       # đường dẫn khi chạy trong wwsl
WIN_REPO_PATH   = r"E:\BD\peft"           # đường dẫn khi chạy trên windows native
REPO_CLONE_URL  = "https://github.com/huggingface/peft.git"

# chọn đường dẫn phù hợp với môi trường đang chạy
if platform.system() == "Windows":
    REPO_ROOT  = Path(WIN_REPO_PATH)
    OUTPUT_DIR = Path(r"E:\BD\output")
else:
    # linux / WSL / macOS
    REPO_ROOT  = Path(WSL_REPO_PATH)
    OUTPUT_DIR = Path("/mnt/e/BD/output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)   # tạo thư mục output nếu chưa có

print(f"Python     : {sys.version}")
print(f"Platform   : {platform.system()} – {platform.version()}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Python     : 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Platform   : Windows – 10.0.26200
REPO_ROOT  : E:\BD\peft
OUTPUT_DIR : E:\BD\output


### 1. Kiểm tra & Clone Repository

In [10]:
def clone_repo_if_needed(repo_root: Path, clone_url: str) -> None:
    """
    Kiểm tra xem repo đã tồn tại tại `repo_root` chưa.
    - Nếu đã có thư mục `.git`-> bỏ qua, in thông báo.
    - Nếu chưa -> thực hiện `git clone --depth 1` (shallow clone để tiết kiệm băng thông).
    """
    git_dir = repo_root / ".git"

    if git_dir.exists():
        print(f"[INFO] Repository đã tồn tại tại: {repo_root}")
        print(f"[INFO] Bỏ qua bước clone.")
        return

    # đảm bảo thư mục cha tồn tại
    repo_root.parent.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Repo chưa tồn tại. Bắt đầu clone (shallow --depth 1)...")
    print(f"[INFO] URL  : {clone_url}")
    print(f"[INFO] Đích : {repo_root}")

    cmd = [
        "git", "clone",
        "--depth", "1",          # shallow clone: chỉ lấy commit mới nhất
        clone_url,
        str(repo_root)
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    # in stdout & stderr để dễ debug
    if result.stdout:
        print("[git stdout]\n" + result.stdout)
    if result.stderr:
        print("[git stderr]\n" + result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"git clone thất bại với exit code {result.returncode}")

    print(f"[OK] Clone thành công -> {repo_root}")


# chạy
clone_repo_if_needed(REPO_ROOT, REPO_CLONE_URL)

# xác nhận thư mục tồn tại sau khi clone  
assert REPO_ROOT.exists(), f"[ERROR] Không tìm thấy repo tại {REPO_ROOT}"
print(f"\n[OK] Repo hợp lệ – thư mục tồn tại: {REPO_ROOT}")

[INFO] Repo chưa tồn tại. Bắt đầu clone (shallow --depth 1)...
[INFO] URL  : https://github.com/huggingface/peft.git
[INFO] Đích : E:\BD\peft
[git stderr]
Cloning into 'E:\BD\peft'...

[OK] Clone thành công -> E:\BD\peft

[OK] Repo hợp lệ – thư mục tồn tại: E:\BD\peft


### 2. Duyệt đệ quy & thu thập tất cả file `.py`

In [11]:
# tìm các file đệ quy
all_py_files_absolute = sorted(REPO_ROOT.rglob("*.py"))

# chuyển sang đường dẫn tương đối để dễ đọc  
all_py_files_relative = [
    p.relative_to(REPO_ROOT)
    for p in all_py_files_absolute
]

total_found = len(all_py_files_relative)
print(f"[OK] Tổng số file .py tìm được: {total_found}")
print("\nVí dụ 10 file đầu tiên:")
for rel_path in all_py_files_relative[:10]:
    print(f"  {rel_path}")

[OK] Tổng số file .py tìm được: 431

Ví dụ 10 file đầu tiên:
  docs\source\_config.py
  examples\adamss_finetuning\glue_adamss_asa_example.py
  examples\adamss_finetuning\glue_adamss_asa_manual_example.py
  examples\adamss_finetuning\image_classification_adamss_asa.py
  examples\adamss_finetuning\test_adamss_quick.py
  examples\alora_finetuning\alora_finetuning.py
  examples\arrow_multitask\arrow_phi3_mini.py
  examples\bdlora_finetuning\chat.py
  examples\beft_finetuning\beft_finetuning.py
  examples\boft_controlnet\__init__.py


### 3. Phân loại file `.py`

**Mục tiêu:** Loại bỏ các file không liên quan trước khi đưa vào CPG Parser, giúp giảm nhiễu
và tăng chất lượng đồ thị phụ thuộc.

**Bốn nhóm phân loại và lý do loại trừ:**

| Nhóm | Tiêu chí | Lý do loại trừ |
|:-----|:---------|:---------------|
| **test** | Thư mục chứa `test`/`tests`; tên file bắt đầu bằng `test_` | Test code không đại diện cho logic nghiệp vụ, chứa mock và fixture gây nhiễu CPG |
| **setup** | Tên file chính xác là `setup.py`, `__init__.py`, `conftest.py` (ngoài thư mục test) | File scaffolding/packaging, không chứa logic domain |
| **auto-generated** | Nằm trong `build/`, `dist/`, `*.egg-info/`; hoặc header chứa `DO NOT EDIT` / `auto-generated` | Code sinh tự động thường bị flatten, không phản ánh cấu trúc thiết kế thực |
| **source** | Tất cả file còn lại | Corpus chính cho CPG Parser Service ở Task 2 |

**Thứ tự ưu tiên phân loại:** `auto-generated` -> `setup` (exact filename) -> `test` (path/prefix) -> `source`

In [12]:
# các hằng số phân loại gồm

# từ khoá xuất hiện trong tên THƯ MỤC → nhóm 'test'
TEST_DIR_KEYWORDS = {"test", "tests"}
# tiền tố tên file -> nhóm 'test' (ví dụ: test_lora.py)
TEST_FILENAME_PREFIX = "test_"

# tên file chính xác -> nhóm 'setup'
# conftest.py đặt ở đây (KHÔNG trong TEST_DIR_KEYWORDS) để tránh bị phân loại sai thành 'test' do 'test' là substring của 'conftest'
SETUP_FILENAMES = {"setup.py", "conftest.py", "__init__.py"}

# thư mục chỉ điểm -> nhóm 'auto-generated'
AUTOGEN_DIRS = {"build", "dist", ".egg-info"}
# marker trong header file -> nhóm 'auto-generated'
AUTOGEN_MARKERS = ("auto-generated", "autogenerated", "do not edit", "do not modify")
# số dòng tối đa đọc để tìm auto-generated marker (tránh đọc file lớn)
HEADER_SCAN_LINES = 10


def is_auto_generated(abs_path: Path) -> bool:
    """
    heuristic phát hiện file auto-generated:
    - Nằm trong thư mục build/, dist/, hoặc *.egg-info/ (kiểm tra theo path parts)
    - Header 10 dòng đầu chứa marker 'DO NOT EDIT' / 'auto-generated'

    nhận abs_path (Path tuyệt đối) để tránh phụ thuộc vào REPO_ROOT global.
    """
    # kiểm tra theo tên thư mục trong đường dẫn
    for part in abs_path.parts:
        if part in AUTOGEN_DIRS or part.endswith(".egg-info"):
            return True

    # kiểm tra nội dung header
    try:
        with open(abs_path, encoding="utf-8", errors="ignore") as fh:
            header_lines = []
            for i, line in enumerate(fh):
                if i >= HEADER_SCAN_LINES:
                    break
                header_lines.append(line.lower())
            header = "".join(header_lines)
        return any(marker in header for marker in AUTOGEN_MARKERS)
    except OSError:
        return False


def classify_file(rel_path: Path) -> str:
    """
    phân loại file python theo 4 nhóm.
    trả về: 'auto-generated' | 'setup' | 'test' | 'source'

    thứ tự ưu tiên phân loại:
      1. auto-generated  – loại trừ trước, ngay cả khi tên file trùng nhóm khác
      2. setup           – khớp CHÍNH XÁC tên file (conftest.py, __init__.py, setup.py)
                           PHẢI kiểm tra TRƯỚC keyword test, vì 'test' là substring
                           của 'conftest' -> nếu đảo thứ tự, conftest.py sẽ bị nhầm
                           thành 'test'
      3. test            – thư mục chứa 'test'/'tests' hoặc tên file bắt đầu 'test_'
      4. source          – tất cả còn lại
    """
    abs_path    = REPO_ROOT / rel_path
    filename    = rel_path.name.lower()
    # Lấy các phần thư mục (bỏ tên file cuối cùng) để kiểm tra keyword
    dir_parts   = [p.lower() for p in rel_path.parts[:-1]]

    # 1. auto-generated  
    if is_auto_generated(abs_path):
        return "auto-generated"

    # 2. setup files
    if filename in SETUP_FILENAMES:
        return "setup"

    # 3. test files 
    if any(kw == part for part in dir_parts for kw in TEST_DIR_KEYWORDS):
        return "test"
    if filename.startswith(TEST_FILENAME_PREFIX):
        return "test"

    # 4. source 
    return "source"


# phân loại
categories = [classify_file(p) for p in all_py_files_relative]

# đếm số lượng từng nhóm
from collections import Counter
category_counts = Counter(categories)

print("=" * 50)
print("KẾT QUẢ PHÂN LOẠI FILE .py")
print("=" * 50)
print(f"  {'test':<18}: {category_counts.get('test', 0):>5} file(s)")
print(f"  {'setup':<18}: {category_counts.get('setup', 0):>5} file(s)")
print(f"  {'auto-generated':<18}: {category_counts.get('auto-generated', 0):>5} file(s)")
print(f"  {'source':<18}: {category_counts.get('source', 0):>5} file(s)")
print("-" * 50)
print(f"  {'TỔNG':<18}: {sum(category_counts.values()):>5} file(s)")

# conftest.py được phân loại đúng là 'setup', không phải 'test'
conftest_files = [(p, c) for p, c in zip(all_py_files_relative, categories)
                  if p.name == "conftest.py"]
print(f"\n── Kiểm tra conftest.py (phải là 'setup', không phải 'test') ────────")
if conftest_files:
    for path, cat in conftest_files[:5]:
        status = "OK" if cat == "setup" else "✗ LỖI"
        print(f"  [{status}] {path}  ->  {cat}")
else:
    print("  (Không tìm thấy conftest.py trong repo)")

KẾT QUẢ PHÂN LOẠI FILE .py
  test              :    63 file(s)
  setup             :    59 file(s)
  auto-generated    :     0 file(s)
  source            :   309 file(s)
--------------------------------------------------
  TỔNG              :   431 file(s)

── Kiểm tra conftest.py (phải là 'setup', không phải 'test') ────────
  [OK] tests\conftest.py  ->  setup


#### 3a. Kiểm chứng heuristic is_auto_generated() (Smoke Test)

Do repository ban đầu không chứa các file sinh tự động nên kết quả phát hiện là 0 file. Để kiểm tra hàm `is_auto_generated()`, nhóm tạo một số file và thư mục giả lập (ví dụ build/, dist/,...) rồi chạy lại chương trình. Nếu các file này được phát hiện đúng là auto-generated thì có thể kết luận detector hoạt động như mong đợi. Sau khi kiểm tra, nhóm xóa các file giả lập để giữ nguyên trạng thái của repository.

In [13]:
# kiểm tra nhanh hàm is_auto_generated()
import tempfile
import textwrap

# trường hợp 1: File có dòng "DO NOT EDIT" -> dự kiến trả về True
with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp:
    tmp.write(textwrap.dedent("""\
        # DO NOT EDIT - This file is auto-generated by the build system.
        # Any manual changes will be overwritten on next build.
        print('hello from generated file')
    """))
    tmp_path = Path(tmp.name)

result_autogen = is_auto_generated(tmp_path)
tmp_path.unlink()   # xóa file sau khi kiểm tra

# trường hợp 2: File Python bình thường -> dự kiến trả về False
with tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp2:
    tmp2.write("def my_func():\n    return 42\n")
    tmp2_path = Path(tmp2.name)

result_normal = is_auto_generated(tmp2_path)
tmp2_path.unlink()

# trường hợp 3: Đường dẫn có thư mục build -> dự kiến trả về True
# không cần tạo file thật vì hàm chỉ kiểm tra tên thư mục
if platform.system() == "Windows":
    fake_build_path = Path(r"C:\project\build\generated_module.py")
else:
    fake_build_path = Path("/project/build/generated_module.py")

result_build = is_auto_generated(fake_build_path)

# hiển thị kết quả
print("=" * 55)
print("Kiểm tra hàm is_auto_generated()")
print("=" * 55)

test_cases = [
    ("File có 'DO NOT EDIT'", result_autogen, True),
    ("File Python bình thường", result_normal, False),
    ("Đường dẫn chứa 'build/'", result_build, True),
]

all_passed = True

for desc, got, expected in test_cases:
    if got == expected:
        print(f"[OK] {desc}")
    else:
        print(f"[Lỗi] {desc} (kết quả: {got}, mong đợi: {expected})")
        all_passed = False

assert all_passed, "Có test chưa đúng, cần kiểm tra lại hàm."

print("\nHoàn thành kiểm tra. Tất cả các trường hợp đều đúng.")

Kiểm tra hàm is_auto_generated()
[OK] File có 'DO NOT EDIT'
[OK] File Python bình thường
[OK] Đường dẫn chứa 'build/'

Hoàn thành kiểm tra. Tất cả các trường hợp đều đúng.


### 4. Tạo DataFrame & Lưu ra CSV

In [14]:
def count_lines(abs_path: Path) -> int:
    """Đếm số dòng của file, nếu không đọc được thì trả về -1."""
    try:
        with open(abs_path, encoding="utf-8", errors="ignore") as fh:
            return sum(1 for _ in fh)
    except OSError:
        return -1


# thu thập thông tin của từng file Python
records = []

for rel_path, category in zip(all_py_files_relative, categories):
    abs_path = REPO_ROOT / rel_path

    try:
        size_bytes = abs_path.stat().st_size
    except OSError:
        size_bytes = -1

    records.append({
        "relative_path": str(rel_path),
        "size_bytes": size_bytes,
        "category": category,
        "num_lines": count_lines(abs_path),
    })

# tạo DataFrame
df = pd.DataFrame(
    records,
    columns=["relative_path", "size_bytes", "category", "num_lines"]
)

# lưu kết quả để dùng cho các phần sau
csv_output_path = OUTPUT_DIR / "discovered_files.csv"
df.to_csv(csv_output_path, index=False, encoding="utf-8")

print(f"Đã lưu file CSV: {csv_output_path}")
print(f"Số lượng dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột\n")

# hiển thị thử vài dòng đầu
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)


print("10 dòng đầu của DataFrame")
print("-" * 60)

df.head(10)

Đã lưu file CSV: E:\BD\output\discovered_files.csv
Số lượng dữ liệu: 431 dòng, 4 cột

10 dòng đầu của DataFrame
------------------------------------------------------------


,relative_path,size_bytes,category,num_lines
0,docs\source\_config.py,287,source,7
1,examples\adamss_finetuning\glue_adamss_asa_example.py,15224,source,382
2,examples\adamss_finetuning\glue_adamss_asa_manual_example.py,16825,source,411
3,examples\adamss_finetuning\image_classification_adamss_asa.py,17368,source,452
4,examples\adamss_finetuning\test_adamss_quick.py,4587,test,176
5,examples\alora_finetuning\alora_finetuning.py,9839,source,259
6,examples\arrow_multitask\arrow_phi3_mini.py,14755,source,383
7,examples\bdlora_finetuning\chat.py,1972,source,64
8,examples\beft_finetuning\beft_finetuning.py,4083,source,122
9,examples\boft_controlnet\__init__.py,0,setup,0


In [15]:
# Thống kê nhanh theo category
summary_df = (
    df.groupby("category")
      .agg(
          num_files   = ("relative_path", "count"),
          total_bytes = ("size_bytes",    "sum"),
          total_lines = ("num_lines",     "sum"),
          avg_lines   = ("num_lines",     "mean"),
      )
      .reset_index()
      .sort_values("num_files", ascending=False)
)
summary_df["avg_lines"] = summary_df["avg_lines"].round(1)

print("Thống kê theo category")
summary_df

Thống kê theo category


,category,num_files,total_bytes,total_lines,avg_lines
1,source,309,4052222,93524,302.7
2,test,63,2065517,48096,763.4
0,setup,59,75268,2144,36.3


### 5. Bảng tổng kết cuối cùng (Lab Report)

In [16]:
# ── Các con số chính cần ghi vào Lab Report ───────────────────────────────
total_py_files = len(df)
n_test         = category_counts.get("test", 0)
n_setup        = category_counts.get("setup", 0)
n_autogen      = category_counts.get("auto-generated", 0)
n_excluded     = n_test + n_setup + n_autogen      # tổng file bị loại trừ
n_source       = category_counts.get("source", 0)  # file đưa vào Parser

print("╔══════════════════════════════════════════════════════╗")
print("║           TASK 1 – FINAL SUMMARY (Lab Report)        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Repository : huggingface/peft                       ║")
print(f"║  URL        : https://github.com/huggingface/peft    ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Total .py files discovered        : {total_py_files:>5}           ║")
print("╠──────────────────────────────────────────────────────╣")
print(f"║  Excluded – test files             : {n_test:>5}           ║")
print(f"║  Excluded – setup/init files       : {n_setup:>5}           ║")
print(f"║  Excluded – auto-generated files   : {n_autogen:>5}           ║")
print(f"║  Total excluded                    : {n_excluded:>5}           ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  SOURCE files -> CPG Parser Service : {n_source:>5}          ║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║           TASK 1 – FINAL SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════╣
║  Repository : huggingface/peft                       ║
║  URL        : https://github.com/huggingface/peft    ║
╠══════════════════════════════════════════════════════╣
║  Total .py files discovered        :   431           ║
╠──────────────────────────────────────────────────────╣
║  Excluded – test files             :    63           ║
║  Excluded – setup/init files       :    59           ║
║  Excluded – auto-generated files   :     0           ║
║  Total excluded                    :   122           ║
╠══════════════════════════════════════════════════════╣
║  SOURCE files -> CPG Parser Service :   309          ║
╚══════════════════════════════════════════════════════╝


### 6. Reflection

**What worked**
- `git clone --depth 1` giúp lấy repo `peft` nhanh và nhẹ, đủ dùng cho việc phân tích tĩnh mã nguồn.
- `pathlib.Path.rglob("*.py")` duyệt đệ quy toàn bộ 431 file `.py` mà không cần xử lý riêng cho từng hệ điều hành.
- Heuristic phân loại 4 nhóm (`auto-generated` -> `setup` -> `test` -> `source`) cho kết quả nhất quán, đã kiểm chứng bằng smoke test riêng cho `is_auto_generated()` (3/3 test case PASS).

**What failed**
- Bản đầu tiên đặt thứ tự ưu tiên là `test` trước `setup`, khiến `conftest.py` bị nhận nhầm thành `test` (do `"test"` là substring của `"conftest"`). Notebook đã bổ sung một bước kiểm tra riêng (in ra danh sách `conftest.py` và category tương ứng) để phát hiện lỗi này.
- Repo `peft` không có sẵn file/thư mục auto-generated (`build/`, `dist/`, `.egg-info/`) nên nhóm không thể xác nhận `is_auto_generated()` hoạt động đúng chỉ bằng cách chạy trên repo thật.

**How resolved**
- Đổi thứ tự ưu tiên phân loại: kiểm tra khớp tên file chính xác (`setup`) trước khi kiểm tra từ khoá `test`, giải quyết dứt điểm lỗi `conftest.py`.
- Viết smoke test tạo file/thư mục giả lập (có marker `DO NOT EDIT`, path chứa `build/`) để kiểm chứng `is_auto_generated()` hoạt động đúng, bù lại việc repo thật không có ca auto-generated nào.